In [ ]:
# ==========================================================
# Marketing A/B Testing Analysis
# Dataset: Marketing_AB.csv
# Goal:
# 1. Analyze whether advertisement increases conversion rate
# 2. Explore the impact of ad exposure frequency
# 3. Analyze time effects (day/hour)
# 4. Build logistic regression model for conversion prediction
# ==========================================================

# =========================
# 1. Import Libraries
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

pd.set_option("display.max_columns", None)

# =========================
# 2. Load Dataset
# =========================
df = pd.read_csv("../data/marketing_AB.csv")
print("Dataset Shape:")
print(df.shape)
print("\nFirst 5 rows:")
print(df.head())

# =========================
# 3. Data Cleaning
# =========================
# 删除CSV保存产生的索引列
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

print("\nColumns:")
print(df.columns)
print("\nData Information:")
df.info()

# 检查缺失值
print("\nMissing Values:")
print(df.isnull().sum())

# 检查重复数据
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# =========================
# 4. Experiment Group EDA
# =========================
# 查看实验组数量
group_count = df["test group"].value_counts()
print("\nExperiment Group Count:")
print(group_count)

# 查看实验组比例
group_ratio = df["test group"].value_counts(normalize=True)
print("\nExperiment Group Ratio:")
print(group_ratio)

# 可视化实验组大小
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="test group")
plt.title("Experiment Group Distribution")
plt.show()

# =========================
# 5. Basic A/B Test Analysis
# =========================
# 每组用户数量
user_count = df.groupby("test group")["user id"].count()
print("\nUser Count:")
print(user_count)

# 每组转化人数
conversion_count = df.groupby("test group")["converted"].sum()
print("\nConversion Count:")
print(conversion_count)

# 计算每组转化率
conversion_rate = df.groupby("test group")["converted"].mean()
print("\nConversion Rate:")
print(conversion_rate)

# 转化率可视化
plt.figure(figsize=(6, 4))
sns.barplot(x=conversion_rate.index, y=conversion_rate.values)
plt.ylabel("Conversion Rate")
plt.title("Conversion Rate Comparison")
plt.show()

# =========================
# 6. Lift Analysis
# =========================
# 计算广告组和PSA组的转化率
ad_rate = conversion_rate["ad"]
psa_rate = conversion_rate["psa"]

# 计算相对提升比例
lift = (ad_rate - psa_rate) / psa_rate
print("\nRelative Lift:")
print(lift)

# =========================
# 7. Chi-square Test
# =========================
# 构造实验组 × 是否转化的列联表
table = pd.crosstab(df["test group"], df["converted"])
print("\nContingency Table:")
print(table)

# 执行卡方独立性检验
chi2, p_value, dof, expected = chi2_contingency(table)
print("\nChi-square:")
print(chi2)
print("\nP-value:")
print(p_value)
print("\nDegree of Freedom:")
print(dof)

# 根据显著性水平判断实验结果
alpha = 0.05
if p_value < alpha:
    print("\nResult: Significant difference exists")
else:
    print("\nResult: No significant difference")

# =========================
# 8. Total Ads Exposure Analysis
# =========================
# 查看广告曝光次数的描述性统计
print("\nTotal Ads Description:")
print(df["total ads"].describe())

# 查看广告曝光次数的分布
plt.figure(figsize=(8, 5))
sns.histplot(df["total ads"], bins=50)
plt.title("Distribution of Total Ads")
plt.show()

# 将曝光次数划分为不同区间
bins = [0, 5, 10, 20, 50, 100, np.inf]
labels = ["0-5", "6-10", "11-20", "21-50", "51-100", "100+"]
df["ads_group"] = pd.cut(df["total ads"], bins=bins, labels=labels)

# 计算不同曝光次数区间的转化率
ads_conversion = df.groupby("ads_group", observed=True)["converted"].mean()
print("\nConversion Rate by Ads Exposure:")
print(ads_conversion)

# 不同曝光次数下的转化率可视化
plt.figure(figsize=(12, 6), dpi=200)
ads_conversion.plot(
    kind="bar",
    edgecolor="white",
    linewidth=0.8
)
plt.ylabel("Conversion Rate", fontsize=9)
plt.title("Conversion Rate by Ads Exposure", fontsize=11, pad=12)
plt.xticks(rotation=45, fontsize=8) # x轴标签旋转45度，防止文字重叠模糊
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# =========================
# 9. Most Ads Day Analysis
# =========================
# 查看用户最常看到广告的星期分布
print("\nAds Day Distribution:")
print(df["most ads day"].value_counts())
# 计算不同星期的转化率
day_conversion = df.groupby("most ads day")["converted"].mean()
print("\nConversion Rate by Day:")
print(day_conversion)
# 设置星期顺序
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]
# 按星期顺序重新排列
day_conversion = day_conversion.reindex(day_order)
print("\nOrdered Conversion Rate by Day:")
print(day_conversion)
# 绘图
plt.figure(figsize=(10,5))

day_conversion.plot(
    kind="bar"
)
plt.xlabel(
    "Day of Week",
    fontsize=12
)
plt.ylabel(
    "Conversion Rate",
    fontsize=12
)
plt.title(
    "Conversion Rate by Day",
    fontsize=14
)
# 调整横坐标字体
plt.xticks(
    rotation=45,
    fontsize=10
)
# 显示数值
for i, v in enumerate(day_conversion):
    plt.text(
        i,
        v + 0.0005,
        f"{v:.2%}",
        ha="center",
        fontsize=10
    )
plt.tight_layout()
plt.show()
# =========================
# 10. Most Ads Hour Analysis
# =========================
# 计算不同小时的转化率
hour_conversion = df.groupby("most ads hour")["converted"].mean()
print("\nConversion Rate by Hour:")
print(hour_conversion)

# 不同小时的转化率可视化
plt.figure(figsize=(10, 4))
hour_conversion.plot(marker="o")
plt.xlabel("Hour")
plt.ylabel("Conversion Rate")
plt.title("Conversion Rate by Hour")
plt.show()

# =========================
# 11. Interaction Analysis
# =========================
# 分析广告类型 × 曝光次数的交互效应
interaction = df.groupby(["test group", "ads_group"], observed=True)["converted"].mean()
print("\nGroup x Ads Exposure Interaction:")
print(interaction)

# 转换为二维表
interaction_table = interaction.unstack()

# 可视化交互效应
interaction_table.plot(kind="bar", figsize=(10, 5))
plt.ylabel("Conversion Rate")
plt.title("Group x Exposure Interaction")
plt.show()

# =========================
# 12. Logistic Regression
# =========================
# 使用广告类型、曝光次数、星期和小时预测用户是否转化
features = ["test group", "total ads", "most ads day", "most ads hour"]
X = df[features]
y = df["converted"]

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 定义类别变量和数值变量
categorical_features = ["test group", "most ads day"]
numeric_features = ["total ads", "most ads hour"]

# 对类别变量进行One-Hot Encoding
# 数值变量保持原始形式
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# 建立Logistic Regression Pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

# 训练模型
model.fit(X_train, y_train)

# 预测类别
y_pred = model.predict(X_test)

# 预测转化概率
y_prob = model.predict_proba(X_test)[:, 1]

# =========================
# 13. Model Evaluation
# =========================
print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

print("\nAUC:")
print(roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
